In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import skrf as rf
from pathlib import Path
from skrf.mathFunctions import mag_2_db

from skrf_usimmics import Substrate, klopfenstein_cpw, klopfenstein_response, usimmics
from skrf_usimmics.plots import COLORS, GUIDE_COLOR, plot_profile, plot_s11

In [ ]:
# CPW geometry and substrate (all lengths in m)
W = 40e-6
S1 = 25e-6
H = 500e-6
EP_R = 11.7

# Taper specification
Z_TARGET = 36.0
F_C = 9e9               # lower passband edge
GAMMA_M_DB = -30.0      # passband reflection ripple
N_SECTIONS = 20

freq = rf.Frequency(0.1, 30, 401, 'GHz')

# Zero-thickness PEC, lossless dielectric, air backside
substrate = Substrate(h=H, ep_r=EP_R)

line = klopfenstein_cpw(W, S1, Z_TARGET, F_C, GAMMA_M_DB, N_SECTIONS, substrate, freq)
report = line.report
Z1 = report['z1']

print(f'Z0(S1 = {S1*1e6:.1f} um) = {Z1:.2f} ohm  (exact 50 ohm needs S = {substrate.gap_for_z0(freq, F_C, W, 50)*1e6:.2f} um)')
print(f'Z0 = {Z_TARGET:.0f} ohm needs S2 = {report["s2"]*1e6:.2f} um')
print(f'Gamma_0 = {report["gamma_0"]:.4f}, rmax = {report["rmax"]:.4f}, A = {report["A"]:.3f}, ep_eff = {report["ep_eff"]:.3f}')
print(f'Taper length L = {line.length*1e3:.2f} mm')

In [ ]:
print(f'Input step: S = {S1*1e6:.2f} -> {line.s[0]*1e6:.2f} um, Z0 = {Z1:.2f} -> {line.z_sections[0]:.2f} ohm')
print(f'Output step: S = {line.s[-2]*1e6:.2f} -> {line.s[-1]*1e6:.2f} um, Z0 = {line.z_sections[-2]:.2f} -> {line.z_sections[-1]:.2f} ohm')

plot_profile(line)
plt.show()

s_table = line.table()
display(s_table.round(3))

In [ ]:
# Port 1 referenced to the feed line, and to 50 ohm. Both include the Klopfenstein end step.
taper = line.network(freq)
taper.name = 'skrf cascade'
taper_50 = line.network(freq, z_ports=(50, Z_TARGET))

# Compare the cascaded network with the analytic Klopfenstein response
fig, ax = plot_s11([taper], 'Klopfenstein taper reflection')
ax.plot(freq.f, mag_2_db(klopfenstein_response(line, freq)), color=COLORS[1], linewidth=2, linestyle='--', label='Analytic')
ax.axhline(GAMMA_M_DB, color=GUIDE_COLOR, linewidth=1, linestyle=':')
ax.axvline(F_C, color=GUIDE_COLOR, linewidth=1, linestyle=':')
ax.legend()
plt.show()

passband = freq.f >= F_C
print(f'Peak |S11| above F_C: {mag_2_db(np.abs(taper.s[passband, 0, 0])).max():.2f} dB')
print(f'Peak |S11| above F_C, port 1 referenced to 50 ohm: {mag_2_db(np.abs(taper_50.s[passband, 0, 0])).max():.2f} dB')
print(f'Min |S21| over sweep: {mag_2_db(np.abs(taper.s[:, 1, 0])).min():.3f} dB')

In [ ]:
# Write the uSimmics schematic, simulate it with the uSimmics engine, and compare with skrf
SCH_PATH = Path.home() / '.qucs' / 'CPW_taper_prj' / 'cpw_taper.sch'

status = usimmics.write_schematic(SCH_PATH, line, freq)
print(f'Schematic {status}: {SCH_PATH}')

taper_usimmics = usimmics.read_sp(usimmics.run_simulation(SCH_PATH), [Z1, Z_TARGET])

plot_s11([taper, taper_usimmics], 'skrf and uSimmics reflection')
plt.show()

print(f'Max |S11 difference|, skrf vs uSimmics: {np.abs(taper.s[:, 0, 0] - taper_usimmics.s[:, 0, 0]).max():.2e}')
print(f'Max |S21 difference|, skrf vs uSimmics: {np.abs(taper.s[:, 1, 0] - taper_usimmics.s[:, 1, 0]).max():.2e}')
print(f'uSimmics peak |S11| above F_C: {mag_2_db(np.abs(taper_usimmics.s[passband, 0, 0])).max():.2f} dB')